# Notebook 03 — Privacy Evaluation
**MRes Computing and Artificial Intelligence — MRES7015**

---
Evaluates three privacy-enhancing mechanisms:
1. **Differential Privacy** (DP-SGD via Opacus) across five epsilon values
2. **Membership Inference Attack** (shadow model method — privacy leakage measurement)
3. **Homomorphic Encryption** overhead (CKKS via TenSEAL)

Run after Notebook 02.


## Stage 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install flwr opacus tenseal -q
print("Libraries ready")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.2/926.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 144.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 80.0 MB/s eta 0:00:00
ERROR: pip's dependency res

In [2]:
import os, json, pickle, warnings
import numpy as np
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from collections import OrderedDict
import tenseal as ts

warnings.filterwarnings('ignore')

DRIVE_BASE  = '/content/drive/MyDrive/FL_Dissertation/'
CLEAN_DIR   = DRIVE_BASE + 'data/cleaned/'
ARTEFACTS   = DRIVE_BASE + 'data/artefacts/'
RESULTS_DIR = DRIVE_BASE + 'results/'
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42

# Load cleaned data
X_train = np.load(CLEAN_DIR + 'X_train.npy')
X_test  = np.load(CLEAN_DIR + 'X_test.npy')
y_train = np.load(CLEAN_DIR + 'y_train.npy')
y_test  = np.load(CLEAN_DIR + 'y_test.npy')

with open(ARTEFACTS + 'metadata.json') as f:
    META_DATA = json.load(f)
INPUT_DIM = META_DATA['input_dim']

# Load partitions
import pickle as pkl

partitions = []
for i in range(10): # Assuming 10 clients from 0 to 9 based on the context
    partition_path = os.path.join(ARTEFACTS, 'fl_partitions', f'mod_client_{i}.pkl')
    try:
        with open(partition_path, 'rb') as f:
            partitions.append(pkl.load(f))
    except FileNotFoundError:
        print(f"Warning: Partition file not found at {partition_path}. Skipping.")
        # If you expect all 10 to be there, you might want to raise an error or handle it differently

print(f"Device:    {DEVICE}")
print(f"Input dim: {INPUT_DIM}")
print(f"Train:     {X_train.shape}  |  Test: {X_test.shape}")
print(f"Loaded {len(partitions)} client partitions.")

Device:    cuda
Input dim: 27
Train:     (436822, 27)  |  Test: (109206, 27)
Loaded 10 client partitions.


In [3]:
class ClinicalNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

print("Model class defined")

Model class defined


## Stage 2 — Differential Privacy Evaluation (ε = 0.5, 1, 2, 5, 10)

In [4]:
def train_with_dp(X_tr, y_tr, epsilon_target, delta=1e-5, epochs=20):
    """Train a local model with differential privacy (DP-SGD via Opacus).
    Returns actual epsilon achieved and AUC-ROC on held-out test set.
    """
    model = ClinicalNN(INPUT_DIM)
    model = ModuleValidator.fix(model)   # Opacus requires no BatchNorm
    model = model.to(DEVICE)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    dataset   = TensorDataset(torch.FloatTensor(X_tr).to(DEVICE),
                               torch.LongTensor(y_tr).to(DEVICE))
    loader    = DataLoader(dataset, batch_size=64, drop_last=True)
    criterion = nn.CrossEntropyLoss()

    privacy_engine = PrivacyEngine()
    model, optimizer, loader = privacy_engine.make_private_with_epsilon(
        module=model, optimizer=optimizer, data_loader=loader,
        epochs=epochs, target_epsilon=epsilon_target,
        target_delta=delta, max_grad_norm=1.0
    )

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    # Evaluate on global test set
    model.eval()
    X_t = torch.FloatTensor(X_test).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(X_t), 1)[:,1].cpu().numpy()
    auc          = roc_auc_score(y_test, probs)
    actual_eps   = privacy_engine.get_epsilon(delta=delta)

    return {
        'target_epsilon': epsilon_target,
        'actual_epsilon': float(actual_eps),
        'auc_roc':        float(auc),
        'delta':          delta,
        'epochs':         epochs
    }

# Use first Trust node for DP evaluation
X_dp, y_dp = partitions[0]

dp_results = []
for eps in [0.5, 1.0, 2.0, 5.0, 10.0]:
    print(f"Training with target ε = {eps}...")
    result = train_with_dp(X_dp, y_dp, epsilon_target=eps, epochs=15)
    dp_results.append(result)
    print(f"  Actual ε: {result['actual_epsilon']:.4f}  |  "
          f"AUC-ROC: {result['auc_roc']:.4f}")

with open(RESULTS_DIR + 'dp_results.json', 'w') as f:
    json.dump(dp_results, f, indent=2)
print("\nDP results saved to Drive")

Training with target ε = 0.5...


  Actual ε: 0.4945  |  AUC-ROC: 0.8762
Training with target ε = 1.0...


  Actual ε: 0.9977  |  AUC-ROC: 0.8341
Training with target ε = 2.0...


  Actual ε: 1.9943  |  AUC-ROC: 0.8340
Training with target ε = 5.0...


  Actual ε: 4.9943  |  AUC-ROC: 0.8125
Training with target ε = 10.0...


  Actual ε: 9.9962  |  AUC-ROC: 0.8215

DP results saved to Drive


## Stage 3 — Membership Inference Attack (Privacy Leakage Measurement)

In [5]:
def membership_inference_attack(model, X_member, y_member,
                                X_nonmember, y_nonmember):
    """Shadow model membership inference attack.

    Measures how well an adversary can distinguish whether a specific
    patient record was used in training. Higher attack AUC = more leakage.

    Interpretation:
      Attack AUC = 0.50  →  No leakage (random chance)
      Attack AUC = 0.55  →  Low risk
      Attack AUC = 0.60  →  Moderate risk
      Attack AUC > 0.65  →  High risk — consider stronger privacy
    """
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')

    def get_loss(X, y):
        X_t = torch.FloatTensor(X).to(DEVICE)
        y_t = torch.LongTensor(y).to(DEVICE)
        with torch.no_grad():
            return criterion(model(X_t), y_t).cpu().numpy()

    # Training records have lower loss than unseen records
    member_loss    = get_loss(X_member,    y_member)
    nonmember_loss = get_loss(X_nonmember, y_nonmember)

    # Attack: label 1 = member, 0 = non-member
    # Use negative loss as score (higher score = more likely member)
    scores = np.concatenate([-member_loss, -nonmember_loss])
    labels = np.concatenate([np.ones(len(member_loss)),
                              np.zeros(len(nonmember_loss))])

    attack_auc = roc_auc_score(labels, scores)
    advantage  = 2 * abs(attack_auc - 0.5)

    if attack_auc < 0.55:   risk = "LOW — acceptable for NHS deployment"
    elif attack_auc < 0.65: risk = "MODERATE — review privacy configuration"
    else:                   risk = "HIGH — strengthen privacy mechanisms"

    return {
        'attack_auc':  float(attack_auc),
        'advantage':   float(advantage),
        'risk_level':  risk,
        'n_members':   len(X_member),
        'n_nonmembers':len(X_nonmember)
    }

# Train a test model on Trust node 0, attack using Trust node 1 as non-member
X_mem,  y_mem  = partitions[0]
X_nmem, y_nmem = partitions[1]
n = min(len(X_mem), len(X_nmem))

test_model = ClinicalNN(INPUT_DIM).to(DEVICE)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_mem).to(DEVICE),
                                  torch.LongTensor(y_mem).to(DEVICE)),
                    batch_size=32, shuffle=True)
opt  = torch.optim.SGD(test_model.parameters(), lr=0.01)
crit = nn.CrossEntropyLoss()
test_model.train()
for _ in range(20):
    for xb, yb in loader:
        opt.zero_grad()
        crit(test_model(xb), yb).backward()
        opt.step()

mia_result = membership_inference_attack(
    test_model, X_mem[:n], y_mem[:n], X_nmem[:n], y_nmem[:n]
)

print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("="*50)
print(f"Attack AUC-ROC:      {mia_result['attack_auc']:.4f}")
print(f"Adversary advantage: {mia_result['advantage']:.4f}")
print(f"Risk level:          {mia_result['risk_level']}")
print(f"(Baseline random chance = 0.5000)")

with open(RESULTS_DIR + 'mia_results.json', 'w') as f:
    json.dump(mia_result, f, indent=2)
print("\nMIA results saved to Drive")

MEMBERSHIP INFERENCE ATTACK RESULTS
Attack AUC-ROC:      0.3983
Adversary advantage: 0.2035
Risk level:          LOW — acceptable for NHS deployment
(Baseline random chance = 0.5000)

MIA results saved to Drive


## Stage 4 — Homomorphic Encryption Overhead (CKKS)

In [6]:
import time

print("CKKS Homomorphic Encryption Overhead Assessment")
print("="*50)

# Create CKKS context
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2**40

# Simulate model parameter vector (subset for feasibility check)
dummy_model = ClinicalNN(INPUT_DIM)
param_vector = np.concatenate([p.detach().numpy().flatten()
                                for p in dummy_model.parameters()])
n_params = len(param_vector)

print(f"Total model parameters: {n_params:,}")
print(f"Testing encryption/decryption overhead...")

he_results = {}
for chunk_size in [1000, 5000, n_params]:
    chunk = param_vector[:min(chunk_size, n_params)].tolist()
    t0 = time.time()
    enc = ts.ckks_vector(context, chunk)
    t_enc = time.time() - t0
    t0 = time.time()
    dec = enc.decrypt()
    t_dec = time.time() - t0
    err = np.max(np.abs(np.array(dec[:len(chunk)]) - np.array(chunk)))
    he_results[f'chunk_{chunk_size}'] = {
        'size': len(chunk),
        'enc_seconds': round(t_enc, 4),
        'dec_seconds': round(t_dec, 4),
        'max_error':   float(err)
    }
    print(f"  Chunk {chunk_size:>7,}: enc={t_enc:.3f}s  "
          f"dec={t_dec:.3f}s  max_error={err:.2e}")

with open(RESULTS_DIR + 'he_results.json', 'w') as f:
    json.dump(he_results, f, indent=2)
print("\nHE results saved to Drive")
print("Notebook 03 COMPLETE — proceed to Notebook 04")

CKKS Homomorphic Encryption Overhead Assessment
Total model parameters: 11,970
Testing encryption/decryption overhead...
  Chunk   1,000: enc=0.007s  dec=0.002s  max_error=6.78e-09
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
  Chunk   5,000: enc=0.014s  dec=0.003s  max_error=7.80e-09
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
  Chunk  11,970: enc=0.021s  dec=0.006s  max_error=7.92e-09

HE results saved to Drive
Notebook 03 COMPLETE — proceed to Notebook 04
